# 🧩 Exercícios — Dijkstra

Nesta atividade, exploramos práticas com o algoritmo de Dijkstra: comparação com Bellman-Ford (em instâncias sem negativos), medição de operações no heap, casos inalcançáveis e reconstrução de caminhos.

In [ ]:
from math import inf
import heapq
from typing import Any, Dict, Tuple, List, Optional
import random
import networkx as nx

def dijkstra_with_counters(G: nx.Graph, s: Any, weight: str='weight'):
    for _, _, data in G.edges(data=True):
        if float(data.get(weight, 1.0)) < 0:
            raise ValueError('Dijkstra requer pesos não negativos')
    p = {v: inf for v in G.nodes()}
    pai = {v: None for v in G.nodes()}
    p[s] = 0.0
    heap = [(0.0, s)]
    visited = set()
    pushes = 1
    pops = 0
    while heap:
        dist_u, u = heapq.heappop(heap)
        pops += 1
        if u in visited:
            continue
        visited.add(u)
        for v, data in G[u].items():
            w = float(data.get(weight, 1.0))
            if p[u] + w < p[v]:
                p[v] = p[u] + w
                pai[v] = u
                heapq.heappush(heap, (p[v], v))
                pushes += 1
    return p, pai, pushes, pops

def bellman_ford_nonneg(G: nx.Graph, s: Any, weight: str='weight'):
    # Versão simples para grafos sem negativos
    d = {v: inf for v in G.nodes()}
    pi = {v: None for v in G.nodes()}
    d[s] = 0.0
    V = list(G.nodes())
    for _ in range(len(V)-1):
        updated = False
        for u, v, data in G.edges(data=True):
            w = float(data.get(weight, 1.0))
            if d[u] + w < d[v]:
                d[v] = d[u] + w
                pi[v] = u
                updated = True
            if d[v] + w < d[u]:
                d[u] = d[v] + w
                pi[u] = v
                updated = True
        if not updated:
            break
    return d, pi

def reconstruct_path(pi: Dict[Any,Any], s: Any, v: Any):
    if v == s:
        return [s]
    if pi.get(v) is None:
        return None
    path = [v]
    while v != s:
        v = pi[v]
        path.append(v)
    path.reverse()
    return path

## 1) Comparar com Bellman-Ford (sem negativos)

In [ ]:
random.seed(7)
G = nx.gnm_random_graph(40, 120, seed=7)
for u, v in G.edges():
    G[u][v]['weight'] = random.randint(1, 20)
p, pai, pushes, pops = dijkstra_with_counters(G, 0)
d_bf, pi_bf = bellman_ford_nonneg(G, 0)
print('Distâncias iguais?', all(abs((p[v] if p[v] < inf else inf) - (d_bf[v] if d_bf[v] < inf else inf)) < 1e-9 for v in G.nodes()))
print('Heap pushes:', pushes, '| pops:', pops)

## 2) Inalcançáveis permanecem +∞

In [ ]:
H = nx.Graph()
H.add_nodes_from(range(6))
H.add_weighted_edges_from([(0,1,2),(1,2,3)])  # componente 0-1-2
# componente 3-4-5 separado
H.add_weighted_edges_from([(3,4,1),(4,5,1)])
p2, pai2, pushes2, pops2 = dijkstra_with_counters(H, 0)
print({v: p2[v] for v in H.nodes()})